In [ ]:
import pandas as pd

In [ ]:
import os

# Change MODEL_NAME to switch between models:
#   'claude-haiku-4-5'
#   'gpt-4o-mini'
#   'gemini-3-1-flash-lite-preview'
MODEL_NAME = 'claude-haiku-4-5'

OUTPUT_FILE = os.path.join(
    os.path.dirname(os.path.abspath('__file__')),
    'output',
    f'combined_{MODEL_NAME}_results_all.csv'
)
df_all = pd.read_csv(OUTPUT_FILE)
print(f'Loaded {len(df_all)} rows for model: {MODEL_NAME}')
print('Columns:', list(df_all.columns))
print()
print('Row counts per task:')
print(df_all['task'].value_counts())

# Task 1 is the binary classification task (yes/no: does this function need exception handling?).
# Tasks 2/3/4 ask for code or exception names — incompatible with yes/no evaluation.
# Filter to task1 only for the binary classification metrics.
df = df_all[df_all['task'] == 'task1'].copy()
print(f'\nUsing task1 only: {len(df)} rows')
print('n_try_except distribution (ground truth):')
print(df['n_try_except'].value_counts())

In [ ]:
def get_first_word_before_comma(text):
    """Parse LLM response to binary: 1 (yes), 0 (no), -1 (unparseable)."""
    if not isinstance(text, str) or not text.strip():
        return -1

    # Find the position of the first comma
    comma_index = text.find(',')

    if comma_index == -1:
        # No comma: take the first word of the whole text
        words = text.strip().split()
        first_word = words[0] if words else ''
    else:
        # Take the first word before the comma
        first_part = text[:comma_index]
        words = first_part.split()
        first_word = words[0] if words else ''

    if first_word.lower() == 'yes':
        return 1
    elif first_word.lower() == 'no':
        return 0
    else:
        return -1

In [ ]:
df['llm_resp_binary'] = df['llm_response'].apply(get_first_word_before_comma)
#df['llm_resp_binary'] = df['llm_response'].apply(lambda x: 1 if 'yes' in x.lower() else 0)

In [ ]:
print('llm_resp_binary distribution:')
print(df['llm_resp_binary'].value_counts())
print()
print('Unparseable rate: {:.1%}'.format((df['llm_resp_binary'] == -1).mean()))
df[df['llm_resp_binary'] == -1].head(2)

In [ ]:
df.groupby(['n_try_except', 'llm_resp_binary']).count()

In [ ]:
df_matrix = df[df['llm_resp_binary'] != -1]
print('Rows used for evaluation (parseable responses):', df_matrix.shape[0])
df_matrix.head(2)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_true = df_matrix['n_try_except']
y_pred = df_matrix['llm_resp_binary']

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f_measure = f1_score(y_true, y_pred, zero_division=0)

# Print metrics
print(f"Overall metrics for model: {MODEL_NAME}")
print(f"Accuracy:  {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F-measure: {f_measure:.2f}")

# Optionally, display the confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)


In [ ]:

metrics_results = []

# Evaluate metrics for each prompt type
for prompt_type in ['style-default', 'style-1-shot', 'style-few-shot', 'style-cot']:
    results = df_matrix[df_matrix['prompt_type'] == prompt_type]

    if results.empty:
        print(f'No parseable data for {prompt_type}, skipping.')
        continue

    y_true = results['n_try_except']
    y_pred = results['llm_resp_binary']

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f_measure = f1_score(y_true, y_pred, zero_division=0)

    # Store the metrics
    metrics_results.append({
        'prompt_type': prompt_type,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f_measure': f_measure
    })

    # Print metrics
    print(f"\nMetrics for {prompt_type} prompt:")
    print(f"Accuracy:  {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print(f"F-measure: {f_measure:.2f}")

    # Confusion Matrix
    cm_pt = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print("\nConfusion Matrix:")
    print(cm_pt)

# Convert the metrics results into a DataFrame and save to CSV
metrics_df = pd.DataFrame(metrics_results)

In [ ]:
metrics_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


# Create a heatmap for the overall confusion matrix
y_true = df_matrix['n_try_except']
y_pred = df_matrix['llm_resp_binary']
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted No', 'Predicted Yes'],
            yticklabels=['Actual No', 'Actual Yes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {MODEL_NAME}')
plt.show()
